# Anime

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
import ast
import plotly.express as px

In [3]:
# load dataset
df_profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")
df_person_anime_works = pd.read_csv("../../data/cleaned/person_anime_works_cleaned.csv")
df_person_details = pd.read_csv("../../data/cleaned/person_details_cleaned.csv")
df_person_alternate_names = pd.read_csv("../../data/cleaned/person_alternate_names_cleaned.csv")



In [4]:
df_person_details["birthday"] = pd.to_datetime(df_person_details["birthday"], errors="coerce")
df_person_details["birth_year"] = df_person_details["birthday"].dt.year.astype("Int64")

df_person_details["favorites"] = pd.to_numeric(df_person_details["favorites"], errors="coerce")


In [5]:
print("Righe totali profiles:", len(df_profiles))
print("Utenti unici (username):", df_profiles["username"].nunique())
print("Location non-missing:", df_profiles["location"].notna().mean().round(3), "→ quota")
print("Joined non-missing:", df_profiles["joined"].notna().mean().round(3), "→ quota")

profiles = df_profiles

Righe totali profiles: 337154
Utenti unici (username): 337154
Location non-missing: 1.0 → quota
Joined non-missing: 0.995 → quota


## In quale aree del mondo gli anime sono maggiormente seguiti?


In [6]:
# per errore che mi dava sotto
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

df = df_profiles.copy()

# extract country from location
profiles["country"] = (
    profiles["location"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
    .str.split(",")
    .str[-1]
    .str.strip()
)

#country names
country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Russia": "Russia",
    "South Korea": "South Korea",
    "Korea, Republic of": "South Korea",
    "Korea, Rep.": "South Korea",

}
profiles["country"] = profiles["country"].replace(country_fix)

profiles["country"] = profiles["country"].where(
    profiles["country"].isna(),
    profiles["country"].str.replace(r"\s+", " ", regex=True).str.title()
)
#unique users per country
country_counts = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .reset_index(name="n_users")
)

#which countries are not recognised by Plotly?
valid_countries = px.data.gapminder()["country"].unique()
not_recognised = (
    country_counts.loc[~country_counts["country"].isin(valid_countries), ["country", "n_users"]]
    .sort_values("n_users", ascending=False)
)
not_recognised.head(30)

# log scale
country_counts["log_users"] = np.log10(country_counts["n_users"] + 1)

# choropleth map
fig = px.choropleth(
    country_counts,
    locations="country",
    locationmode="country names",
    color="log_users",
    hover_name="country",
    hover_data={"n_users": True, "log_users": ":.2f"},
    title="Dove gli anime sono maggiormente seguiti? (utenti per paese, scala log10)",
)

fig.update_layout(
    coloraxis_colorbar=dict(title="log10(utenti)", tickformat=".1f")
)

fig.update_geos(showland=True, landcolor="lightgrey", showocean=True, oceancolor="#e6f2ff")

#Layout
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()


La mappa evidenzia paesi in cui il dataset registra una maggiore presenza di utenti appassionati di anime. L'intensità del colore cresce all'aumentare del numero di utenti unici per paese; l'uso della trasformazione log(n_users + 1) comprime le differenze estreme di valore e rende visibili anche paesi con valori più bassi.

#### A questo punto possiamo vedere come l'interesse per gli anime si è sviluppato durante il corso del tempo, considerando i paesi dove esse son maggiormente diffuse.

## Come si è evoluta nel tempo la crescita della community/nuovi utenti per gli anime nelle aree dove sono più concentrate?

In [7]:
df = df_profiles.copy()

# converto joined + extract year
profiles["joined"] = pd.to_datetime(profiles["joined"], errors="coerce")
profiles["joined_year"] = profiles["joined"].dt.year

#filter year
profiles = profiles[(profiles["joined_year"] >= 1917) & (profiles["joined_year"] <= 2024)]

#Top 6 countries
top_countries = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .sort_values(ascending=False)
            .head(6)
            .index
)
profiles_top = profiles[profiles["country"].isin(top_countries)].copy()
# new users to year
trend = (
    profiles_top.dropna(subset=["joined_year", "country"])
                .groupby(["joined_year", "country"])["username"]
                .nunique()
                .reset_index(name="n_users")
)

fig = px.line(trend, x="joined_year", y="n_users", color="country", markers=True)

#line chart
fig = px.line(
    trend,
    x="joined_year",
    y="n_users",
    color="country",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno (top paesi)"
)

fig.update_layout(
    template="plotly_white",
    width=1100, height=600,
    xaxis_title="Anno",
    yaxis_title="Nuovi utenti" ,
    legend_title_text = "Paese"
)

fig.show()


Il grafico mostra l'evoluzione nel tempo dei nuovi utenti registrati ogni anno nei 6 paesi con più utenti nel dataset;


#### E l'evoluzione a livello globale? Vediamo come si modifica a livello temporale ...

## Crescita della community nel tempo: nuovi utenti registrati per anno

In [8]:
df = df_profiles.copy()
#convert joined to datetime and extract  year
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

users_per_year = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)["username"]
      .nunique()
      .reset_index(name="new_users")
      .sort_values("joined_year")
)

users_per_year = users_per_year[users_per_year["joined_year"].between(2005, 2024)].copy()

#line chart
fig = px.line(
    users_per_year,
    x="joined_year",
    y="new_users",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno",
    labels={"joined_year": "Anno", "new_users": "Nuovi utenti"},
    color_discrete_sequence=["#1f4e79"]
)
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()

Il grafico mostra l’andamento dei nuovi utenti registrati ogni anno: si osserva una crescita nel tempo, con picchi marcati negli anni più recenti.

Dopo aver osservato la crescita della community nel tempo, analizziamo come sono composte le liste degli utenti, distinguendo  l'anno di registrazione, per capire se le abitudini di consumo variano tra utenti ‘vecchi’ e ‘nuovi’.

## Composizione delle liste per anno di iscrizione:

In [9]:
df = df_profiles.copy()
# anno iscrizione
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

# colonne di consumo
cols = ["watching", "completed", "on_hold", "dropped", "plan_to_watch"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# aggregazione per anno
consumption_time = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)[cols]
      .sum()
      .reset_index()
      .sort_values("joined_year")
)
#periodo
consumption_time = consumption_time[consumption_time["joined_year"].between(2005, 2024)].copy()

#trasformo in percentuale, così da avere dati più omogenei
den = consumption_time[cols].sum(axis=1)
consumption_pct = consumption_time.copy()
consumption_pct[cols] = consumption_pct[cols].div(den, axis=0) * 100

#grafico
consumption_long = consumption_pct.melt(
    id_vars="joined_year",
    value_vars=cols,
    var_name="status",
    value_name="share"
)

fig = px.area(
    consumption_long,
    x="joined_year",
    y="share",
    color="status",
    title="Composizione delle liste per coorte di iscrizione",
    labels={"joined_year": "Anno di iscrizione", "share": "Quota (%)", "status": "Status"}
)

fig.update_layout(template="plotly_white", width=1100, height=600)
fig.update_yaxes(ticksuffix="%", range=[0,100], dtick=10)
fig.update_traces(hovertemplate="Quota=%{y:.1f}%<br>Anno=%{x}<extra></extra>")
fig.show()

Il grafico mostra, per ogni coorte di iscrizione (anno di registrazione), la composizione percentuale delle liste degli utenti nei diversi status (watching, completed, on_hold, dropped, plan_to_watch). Le quote sommano a 100% per anno e permettono di confrontare come cambia la struttura delle liste tra coorti.